# 3 · Native continuum Sobolev SOLA in pygeoinf0

Use the **same H2 space, three targets, exact constant response and normalized
penalty** as notebook 2. Define G and T as bounded functionals on H, then assemble
$GG^*$ and $TG^*$ directly. No model trial dimension is introduced.

“Continuum” means the mathematical formulation and its data/target representation.
Numerical kernel pairings still need integration and linear solves. Here refinement
means more accurate pairings, not more model coefficients.

In [ ]:
%matplotlib inline
from pathlib import Path
import hashlib, json, os, socket, subprocess, sys, time
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import study as F
S=F.S
ex=F.Experiment(length_scale=.1)  # shared by both H2 routes; edit consistently in all notebooks
OUT=Path('output');OUT.mkdir(exist_ok=True)
COLORS=('#2563eb','#dc2626','#7c3aed')
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':.2,'font.size':11})
print('host:',socket.gethostname(),'Python:',sys.executable)
print('source commit:',subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())
print('original problem SHA256:',hashlib.sha256(Path(S.P.__file__).read_bytes()).hexdigest())
print('600 original rays; one fixed noise draw; sigma =',S.P.SIGMA)
for label,c,t in zip(F.LABELS,ex.centres,ex.truth):print(label,c,'radius',ex.radius,'true disk mean',f'{t:.9f}')
print('Fixed normalized trade-off lambda =',ex.lam,'; H2 length scale =',ex.length_scale)
import platform, scipy, pygeoinf0 as pg
from threadpoolctl import threadpool_info
runtime=dict(host=socket.gethostname(),python=sys.version,numpy=np.__version__,scipy=scipy.__version__,
             platform=platform.platform(),cpu_count=os.cpu_count(),package_source=pg.__file__,
             threadpools=threadpool_info(),targets=ex.centres,radius=ex.radius,lambda_value=ex.lam,length_scale=ex.length_scale,
             study_sha256=hashlib.sha256(Path(F.__file__).read_bytes()).hexdigest(),
             helper_sha256=hashlib.sha256(Path(S.__file__).read_bytes()).hexdigest(),
             problem_sha256=hashlib.sha256(Path(S.P.__file__).read_bytes()).hexdigest())
F.save(OUT/'runtime.json',runtime)
print('BLAS thread counts:',[v['num_threads'] for v in runtime['threadpools']])


## Operators first

Riesz representation gives $g_i=\mathcal R_H^{-1}G_i$ and
$t_a=\mathcal R_H^{-1}T_a$. Then
$$K_{ij}=\langle g_i,g_j\rangle_H,\qquad
B_{ai}=\langle t_a,g_i\rangle_H.$$
In `pygeoinf0`, the direct construction is:

In [ ]:
import pygeoinf0 as pg
H,F_ray,F_target=F.sobolev_objects(ex,32)
with S.quadrature_nodes(24):
    G=pg.Functionals(F_ray,name='G')
    T=pg.Functionals(F_target,name='T')
    K=(G@G.adjoint).matrix()
    B=(T@G.adjoint).matrix()
    tt=np.diag(T.gram())
tau=ex.lam*tt[0]
X=F.constrained_sola(K,B,tau,ex.v)
print('Data-space Gram:',K.shape,'target/data pairing:',B.shape,'X:',X.shape)
print('Constant responses:',X@ex.v)
print('Estimates:',X@ex.d,'actual delivered truths:',X@ex.d0,'target truths:',ex.truth)
# A = X G is an actual operator on the continuum H2 space.
A=pg.Matrix(X,domain=G.codomain,codomain=T.codomain)@G
print('Delivered operator:',A)


## Refine integration and check target quadrature separately

The ray-pair quadrature is varied at fixed 32-chord disk quadrature. We then check
32 versus 64 disk chords separately, with the penalty held fixed. Agreement with
another integration rule is a numerical sensitivity check, not a rigorous bound.

The same positive penalty and response constraint are retained. No clipping of
negative Gram eigenvalues is used to manufacture a continuum Gram.

In [ ]:
ref,Xref,tau,reference=F.native_reference(ex,24,32)
rows=[];weights=[]
for nodes in (10,16,24):
    system=F.native_system(ex,nodes,32)
    low=float(la.eigvalsh(system['K'],subset_by_index=[0,0])[0])
    if low < -1e-10*la.norm(ref['K'],2):raise ArithmeticError('under-resolved Gram')
    X,row=F.solve_system(system,ex,tau)
    row.update(nodes=nodes,min_eigenvalue=low,**F.compare_to_reference(X,ref,Xref))
    row['gram_error']=float(la.norm(system['K']-ref['K'])/la.norm(ref['K']))
    rows.append(row);weights.append(X)
    print('nodes',nodes,'relative X error',row['weight_error'],'estimates',row['estimate'])
fine=F.native_system(ex,24,64)
X64=F.constrained_sola(fine['K'],fine['B'],tau,ex.v)
target_check=dict(target_chords=(32,64),penalty_fixed=tau,
                  relative_X_change=float(la.norm(X64-Xref)/la.norm(Xref)))
print('Separate target quadrature check:',target_check)
F.save(OUT/'continuum.json',dict(rows=rows,reference=reference,tau=tau,target_check=target_check))
np.savez_compressed(OUT/'continuum_reference.npz',K=ref['K'],B=ref['B'],tt=ref['tt'],X=Xref)
assert rows[1]['weight_error']<1e-7
assert target_check['relative_X_change']<1e-3


In [ ]:
nodes=np.array([r['nodes'] for r in rows])
fig,ax=plt.subplots(1,2,figsize=(10,3.8),constrained_layout=True)
for key,label in [('gram_error','K'),('weight_error','X')]:
    ax[0].semilogy(nodes,np.maximum([r[key] for r in rows],1e-16),'o-',label=label)
ax[0].legend();ax[0].set_xlabel('ray quadrature nodes');ax[0].set_title('Reference: 24 nodes / 32 chords')
for k,(color,label) in enumerate(zip(COLORS,F.LABELS)):
    ax[1].plot(nodes,[r['estimate'][k] for r in rows],'o-',color=color,label=label)
    ax[1].axhline(ex.truth[k],color=color,ls=':')
ax[1].legend(fontsize=8);ax[1].set_xlabel('ray quadrature nodes');ax[1].set_title('Target mismatch survives convergence')
plt.show()


## Riesz functions are not Lebesgue averaging-density kernels

The delivered representer is $r_H=G^*x$, so the delivered property is
$\langle r_H,m\rangle_H=\sum_i x_i\int_{\Gamma_i}m\,ds$.
A smooth plot of $r_H$ does **not** mean that the property is $\int r_Hm\,dA$.
The physical observation kernels remain ray measures; the Sobolev metric makes
the functionals bounded and supplies their Riesz representatives.

In [ ]:
r=ref['G'].adjoint(ref['G'].codomain(Xref[0]));t=ref['T'][0]
z=(np.arange(60)+.5)/60;xx,yy=np.meshgrid(z,z)
with S.quadrature_nodes(24):rv,tv=r(xx,yy),t(xx,yy)
fig,ax=plt.subplots(1,2,figsize=(10,4),constrained_layout=True)
for a,v,title in zip(ax,[tv,rv],['Target Riesz function','Delivered Riesz function']):
    im=a.imshow(v,origin='lower',extent=[0,1,0,1],cmap='viridis')
    a.add_patch(plt.Circle(ex.centres[0],ex.radius,fill=False,color='white',lw=1))
    a.set_title(title+' (H2 pairing)');a.grid(False);fig.colorbar(im,ax=a,shrink=.8)
plt.show()
for k,label in enumerate(F.LABELS):
    print(label,'target:',ex.truth[k],'delivered:',reference['delivered_truth'][k],
          'estimate:',reference['estimate'][k],'noise sigma:',reference['std'][k])


## Read the Riesz plots as responses to test anomalies

The top row shows the physical measures: an exact ray integral, a uniform disk
average, and the signed weighted rays selected by SOLA. A ray has zero area but
nonzero measure mass. The disk has density $1/(\pi R^2)$ inside its boundary.
The delivered measure has mass $\sum_i x_iL_i=1$ and retains its ray support.

For the bottom row, place the **same known unit-peak smooth test anomaly** at each
centre $z$ and ask what these physical measurements return:
$$G_iq_z,\qquad T_aq_z,\qquad (XG)_a q_z.$$
The test shape and its horizontal profile are shown explicitly. Its chosen
length is 0.1 throughout; it is neither a physical ray width nor information
about the unknown model. It has decaying tails, rather than compact support.
The heatmaps are **responses to test models**, not area-density representations
of the ray measures. Ray responses have units of model times length; target and
delivered averages have model units, with a shared colour scale for that pair.

Here the common test anomaly is $q_z=K_H(\cdot,z)/K_H(z,z)$. Reproduction gives
$$r_F(z)=F(K_H(\cdot,z)),\qquad F(q_z)=r_F(z)/K_H(z,z).$$
Thus a Riesz image, rescaled by the constant kernel peak, already is a physical
response map for this family of known models. It is not a blurred observation.
The same statement applies to sensitivity, target and resolving functionals.
The requested and delivered maps below use the same probe for all three disks;
any discrepancy is an actual target discrepancy for that test model, rather than
a universal error bound for an unrestricted unknown model.

In [ ]:
# physical-probe: native
import probe_views as V
probes=V.response_grid(ex,OUT)
V.plot_probe(ex,probes);plt.show()
V.plot_physical_and_responses(ex,probes,Xref[0],
    'Native H2 SOLA: exact measures and their common-probe responses');plt.show()
V.plot_three_targets(ex,probes,Xref);plt.show()
print('Kernel peak used for unit-peak normalization:',float(probes['peak']))
print('Unit response to constant models:',Xref@ex.v)


## The two Sobolev routes solve the same limiting problem

For $P_h$ the conforming H-orthogonal trial projector,
$$K_h=GP_hG^*\to GG^*,\qquad B_h=TP_hG^*\to TG^*.$$
The finite Galerkin weights approach the native continuum weights. Numerical
quadrature must also converge; the two limits are separate.

Both calculations select a delivered question $XG$. Neither has proved exact
recovery of the physical target T or a target-error bound without model information.
The advantage established here is a well-defined, stable question-selection
problem under refinement. Notebook 4 measures the cost of these realizations.